# soxlbacktest_qlib_walkforward_baseline_v1

Purpose: convert the existing `soxl_backtest_policy_reimplementation_v1` reference into qlib-calendar-aligned walk-forward evidence, then screen a few risk-capped parameter variants.

Governance:
- Canonical reference baseline: `soxl_backtest_policy_reimplementation_v1`.
- Direct comparison baseline: `legacy_default` fresh-start replay on the qlib walk-forward test windows.
- Role: `diagnose` / `reference`; this card does not promote a model.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus `ytd_2026` for current SOXLBacktest/current-best alignment.
- Changed layer: execution / position sizing parameters inside the legacy SOXLBacktest policy replay.
- Hypothesis: SOXLBacktest can provide a qlib-aligned buy/sell reference; pure risk caps may satisfy the <20% walk-forward drawdown gate but are likely to fail the return-alignment gate.


In [1]:

from __future__ import annotations

import ast
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.policy.legacy_soxl import replay_legacy_soxl_policy

CARD = ROOT / "runtime" / "model_selection" / "card" / "soxlbacktest_qlib_walkforward_baseline_v1"
CARD.mkdir(parents=True, exist_ok=True)
ARTIFACTS = {
    "candidate_metrics": CARD / "candidate_metrics.csv",
    "candidate_gate_summary": CARD / "candidate_gate_summary.csv",
    "candidate_daily": CARD / "candidate_daily.csv",
    "candidate_trades": CARD / "candidate_trades.csv",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


print({"card": rel(CARD), "artifact_count": len(ARTIFACTS)})


{'card': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1', 'artifact_count': 6}


In [2]:

def load_walk_forward_windows(path: Path) -> pd.DataFrame:
    module = ast.parse(path.read_text(encoding="utf-8"))
    for node in module.body:
        if isinstance(node, ast.Assign):
            targets = node.targets
            value_node = node.value
        elif isinstance(node, ast.AnnAssign):
            targets = [node.target]
            value_node = node.value
        else:
            continue
        for target in targets:
            if isinstance(target, ast.Name) and target.id == "WALK_FORWARD_WINDOWS":
                value = ast.literal_eval(value_node)
                rows = []
                for name, train_seg, valid_seg, test_seg in value:
                    rows.append({
                        "window": name,
                        "train_start": train_seg[0],
                        "train_end": train_seg[1],
                        "valid_start": valid_seg[0],
                        "valid_end": valid_seg[1],
                        "start": test_seg[0],
                        "end": test_seg[1],
                        "split_type": "walk_forward_test",
                    })
                return pd.DataFrame(rows)
    raise ValueError(f"WALK_FORWARD_WINDOWS not found in {path}")


wf_windows = load_walk_forward_windows(ROOT / "src" / "soxl_quant" / "app" / "workers.py")
ytd_window = pd.DataFrame([
    {
        "window": "ytd_2026",
        "train_start": "",
        "train_end": "",
        "valid_start": "",
        "valid_end": "",
        "start": "2026-01-01",
        "end": "2026-06-09",
        "split_type": "current_ytd_reference",
    }
])
eval_windows = pd.concat([wf_windows, ytd_window], ignore_index=True)
qlib_calendar = pd.Index(
    pd.to_datetime((ROOT / "data" / "qlib" / "soxl" / "calendars" / "day.txt").read_text().splitlines())
)
eval_windows


,window,train_start,train_end,valid_start,valid_end,start,end,split_type
0,wf_2020,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,walk_forward_test
1,wf_2022,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,walk_forward_test
2,wf_2023_2024,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,walk_forward_test
3,ytd_2026,,,,,2026-01-01,2026-06-09,current_ytd_reference


In [3]:

handoff_path = ROOT / "runtime" / "model_selection" / "card" / "retraining_gate_gap_handoff_v1" / "training_objective_handoff.json"
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
required_soxlbacktest_return_pct = float(handoff["required_soxlbacktest_return_pct"])
required_best_return_pct = float(handoff["required_best_return_pct"])
max_walkforward_drawdown_pct = float(handoff["retention_gate"]["max_walkforward_drawdown_pct"])
required_wf_windows = list(handoff["retention_gate"]["required_wf_windows"])
thresholds = {
    "required_soxlbacktest_return_pct": required_soxlbacktest_return_pct,
    "required_best_return_pct": required_best_return_pct,
    "max_walkforward_drawdown_pct": max_walkforward_drawdown_pct,
    "required_wf_windows": required_wf_windows,
}
thresholds


{'required_soxlbacktest_return_pct': 132.077978,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [4]:

CANDIDATES: list[dict[str, Any]] = [
    {
        "candidate": "legacy_default",
        "role": "buy_sell_reference",
        "config_overrides": {},
        "note": "Exact SOXLBacktest policy reimplementation used by the dashboard reference baseline.",
    },
    {
        "candidate": "risk_cap_35_15_05",
        "role": "medium_risk_probe",
        "config_overrides": {
            "max_position": 0.35,
            "min_position": 0.0,
            "buy_step_initial": 0.15,
            "buy_step_normal": 0.05,
        },
        "note": "First probe that makes wf_2020 drawdown pass, but still carries too much 2022 bear exposure.",
    },
    {
        "candidate": "low_risk_20_08_02",
        "role": "near_drawdown_boundary_probe",
        "config_overrides": {
            "max_position": 0.20,
            "min_position": 0.0,
            "buy_step_initial": 0.08,
            "buy_step_normal": 0.02,
        },
        "note": "Narrower buy sizing near the 2022 drawdown boundary.",
    },
    {
        "candidate": "wf_pass_ytd18_15_15_02",
        "role": "wf_drawdown_pass_best_ytd_probe",
        "config_overrides": {
            "max_position": 0.15,
            "min_position": 0.0,
            "buy_step_initial": 0.15,
            "buy_step_normal": 0.02,
            "sell_score_threshold": 70.0,
            "take_profit_threshold": 0.2,
        },
        "note": "Best YTD probe found in the low-position WF drawdown-pass screen; still far below return-alignment gates.",
    },
    {
        "candidate": "low_risk_15_05_02",
        "role": "wf_drawdown_pass_low_return_floor",
        "config_overrides": {
            "max_position": 0.15,
            "min_position": 0.0,
            "buy_step_initial": 0.05,
            "buy_step_normal": 0.02,
        },
        "note": "All three walk-forward MaxDD values pass <20%, but current YTD return is too low.",
    },
    {
        "candidate": "floor_10_03_01",
        "role": "very_low_risk_floor",
        "config_overrides": {
            "max_position": 0.10,
            "min_position": 0.0,
            "buy_step_initial": 0.03,
            "buy_step_normal": 0.01,
        },
        "note": "Risk floor for confirming the return/drawdown trade-off shape.",
    },
]

pd.DataFrame([
    {"candidate": item["candidate"], "role": item["role"], "config_overrides": json.dumps(item["config_overrides"], sort_keys=True), "note": item["note"]}
    for item in CANDIDATES
])


,candidate,role,config_overrides,note
0,legacy_default,buy_sell_reference,{},Exact SOXLBacktest policy reimplementation use...
1,risk_cap_35_15_05,medium_risk_probe,"{""buy_step_initial"": 0.15, ""buy_step_normal"": ...","First probe that makes wf_2020 drawdown pass, ..."
2,low_risk_20_08_02,near_drawdown_boundary_probe,"{""buy_step_initial"": 0.08, ""buy_step_normal"": ...",Narrower buy sizing near the 2022 drawdown bou...
3,wf_pass_ytd18_15_15_02,wf_drawdown_pass_best_ytd_probe,"{""buy_step_initial"": 0.15, ""buy_step_normal"": ...",Best YTD probe found in the low-position WF dr...
4,low_risk_15_05_02,wf_drawdown_pass_low_return_floor,"{""buy_step_initial"": 0.05, ""buy_step_normal"": ...","All three walk-forward MaxDD values pass <20%,..."
5,floor_10_03_01,very_low_risk_floor,"{""buy_step_initial"": 0.03, ""buy_step_normal"": ...",Risk floor for confirming the return/drawdown ...


In [5]:

def qlib_alignment_for_daily(daily: pd.DataFrame, start: str, end: str) -> dict[str, Any]:
    if daily.empty:
        return {
            "qlib_calendar_aligned": False,
            "missing_qlib_dates": 0,
            "first_date": "",
            "last_date": "",
            "within_requested_window": False,
        }
    dates = pd.to_datetime(daily["date"])
    missing = dates[~dates.isin(qlib_calendar)]
    return {
        "qlib_calendar_aligned": bool(missing.empty),
        "missing_qlib_dates": int(len(missing)),
        "first_date": dates.min().date().isoformat(),
        "last_date": dates.max().date().isoformat(),
        "within_requested_window": bool(dates.min() >= pd.Timestamp(start) and dates.max() <= pd.Timestamp(end)),
    }


metric_rows: list[dict[str, Any]] = []
daily_frames: list[pd.DataFrame] = []
trade_frames: list[pd.DataFrame] = []

for candidate in CANDIDATES:
    name = candidate["candidate"]
    overrides = candidate["config_overrides"]
    for _, window in eval_windows.iterrows():
        result = replay_legacy_soxl_policy(
            start=str(window["start"]),
            end=str(window["end"]),
            config_overrides=overrides,
            root=ROOT,
        )
        alignment = qlib_alignment_for_daily(result.daily, str(window["start"]), str(window["end"]))
        metric_rows.append({
            "candidate": name,
            "role": candidate["role"],
            "window": str(window["window"]),
            "split_type": str(window["split_type"]),
            "train_start": str(window["train_start"]),
            "train_end": str(window["train_end"]),
            "valid_start": str(window["valid_start"]),
            "valid_end": str(window["valid_end"]),
            "requested_start": str(window["start"]),
            "requested_end": str(window["end"]),
            "config_overrides": json.dumps(overrides, sort_keys=True),
            **result.metrics,
            **alignment,
        })
        daily = result.daily.copy()
        daily.insert(0, "window", str(window["window"]))
        daily.insert(0, "candidate", name)
        daily["close"] = daily["price"]
        daily_frames.append(daily)
        trades = result.trades.copy()
        trades.insert(0, "window", str(window["window"]))
        trades.insert(0, "candidate", name)
        trade_frames.append(trades)

metrics = pd.DataFrame(metric_rows)
daily_all = pd.concat(daily_frames, ignore_index=True)
trades_all = pd.concat(trade_frames, ignore_index=True)
metrics


,candidate,role,window,split_type,train_start,train_end,valid_start,valid_end,requested_start,requested_end,...,rulebase_return_pct,rulebase_max_drawdown_pct,average_position_pct,end_position_pct,trade_count,qlib_calendar_aligned,missing_qlib_dates,first_date,last_date,within_requested_window
0,legacy_default,buy_sell_reference,wf_2020,walk_forward_test,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,...,220.777205,54.275144,79.102332,98.13,4,True,0,2020-01-02,2020-12-31,True
1,legacy_default,buy_sell_reference,wf_2022,walk_forward_test,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,...,-79.396371,84.918053,86.692988,90.87,8,True,0,2022-01-03,2022-12-30,True
2,legacy_default,buy_sell_reference,wf_2023_2024,walk_forward_test,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,...,2.935118,32.461810,31.043492,68.71,4,True,0,2024-01-02,2024-12-31,True
3,legacy_default,buy_sell_reference,ytd_2026,current_ytd_reference,,,,,2026-01-01,2026-06-09,...,132.077977,13.554089,16.220734,15.91,3,True,0,2026-01-02,2026-06-09,True
4,risk_cap_35_15_05,medium_risk_probe,wf_2020,walk_forward_test,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,...,69.604552,17.942350,31.031976,55.58,4,True,0,2020-01-02,2020-12-31,True
5,risk_cap_35_15_05,medium_risk_probe,wf_2022,walk_forward_test,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,...,-37.953340,43.271216,27.872112,32.27,10,True,0,2022-01-03,2022-12-30,True
6,risk_cap_35_15_05,medium_risk_probe,wf_2023_2024,walk_forward_test,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,...,4.407749,2.628936,2.790317,9.48,5,True,0,2024-01-02,2024-12-31,True
7,risk_cap_35_15_05,medium_risk_probe,ytd_2026,current_ytd_reference,,,,,2026-01-01,2026-06-09,...,18.308910,0.777618,1.785046,14.89,3,True,0,2026-01-02,2026-06-09,True
8,low_risk_20_08_02,near_drawdown_boundary_probe,wf_2020,walk_forward_test,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,...,35.609710,9.073791,17.395810,35.58,5,True,0,2020-01-02,2020-12-31,True
9,low_risk_20_08_02,near_drawdown_boundary_probe,wf_2022,walk_forward_test,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,...,-21.715680,24.732060,13.705139,12.60,11,True,0,2022-01-03,2022-12-30,True


In [6]:

def candidate_gate_summary(metrics: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for candidate, group in metrics.groupby("candidate", sort=False):
        wf = group[group["split_type"].eq("walk_forward_test")]
        ytd = group[group["window"].eq("ytd_2026")]
        ytd_row = ytd.iloc[0] if not ytd.empty else None
        windows_present = sorted(wf["window"].astype(str).unique().tolist())
        missing = sorted(set(required_wf_windows) - set(windows_present))
        worst_wf_dd = float(wf["rulebase_max_drawdown_pct"].max()) if not wf.empty else float("nan")
        min_wf_return = float(wf["rulebase_return_pct"].min()) if not wf.empty else float("nan")
        mean_wf_return = float(wf["rulebase_return_pct"].mean()) if not wf.empty else float("nan")
        ytd_return = float(ytd_row["rulebase_return_pct"]) if ytd_row is not None else float("nan")
        ytd_dd = float(ytd_row["rulebase_max_drawdown_pct"]) if ytd_row is not None else float("nan")
        wf_drawdown_pass = bool(missing == [] and worst_wf_dd < max_walkforward_drawdown_pct)
        return_eps = 1e-6
        ytd_soxlbacktest_pass = bool(ytd_return + return_eps >= required_soxlbacktest_return_pct)
        ytd_best_pass = bool(ytd_return + return_eps >= required_best_return_pct)
        qlib_aligned = bool(group["qlib_calendar_aligned"].all() and group["within_requested_window"].all())
        failures: list[str] = []
        failures.extend(f"missing_wf_window:{item}" for item in missing)
        if not qlib_aligned:
            failures.append("qlib_calendar_alignment_failed")
        if not wf_drawdown_pass:
            failures.append("worst_wf_max_drawdown_ge_20")
        if not ytd_soxlbacktest_pass:
            failures.append("ytd_return_below_soxlbacktest_reference")
        if not ytd_best_pass:
            failures.append("ytd_return_below_current_best_alignment")
        rows.append({
            "candidate": candidate,
            "role": str(group.iloc[0]["role"]),
            "config_overrides": str(group.iloc[0]["config_overrides"]),
            "wf_windows_present": ",".join(windows_present),
            "missing_wf_windows": ";".join(missing),
            "worst_wf_max_drawdown_pct": round(worst_wf_dd, 6),
            "min_wf_return_pct": round(min_wf_return, 6),
            "mean_wf_return_pct": round(mean_wf_return, 6),
            "ytd_return_pct": round(ytd_return, 6),
            "ytd_max_drawdown_pct": round(ytd_dd, 6),
            "wf_drawdown_pass": wf_drawdown_pass,
            "ytd_soxlbacktest_return_pass": ytd_soxlbacktest_pass,
            "ytd_current_best_alignment_pass": ytd_best_pass,
            "qlib_calendar_aligned": qlib_aligned,
            "retention_gate_pass": bool(wf_drawdown_pass and ytd_soxlbacktest_pass and ytd_best_pass and qlib_aligned),
            "failures": ";".join(failures),
        })
    return pd.DataFrame(rows)


gate_summary = candidate_gate_summary(metrics)
gate_summary


,candidate,role,config_overrides,wf_windows_present,missing_wf_windows,worst_wf_max_drawdown_pct,min_wf_return_pct,mean_wf_return_pct,ytd_return_pct,ytd_max_drawdown_pct,wf_drawdown_pass,ytd_soxlbacktest_return_pass,ytd_current_best_alignment_pass,qlib_calendar_aligned,retention_gate_pass,failures
0,legacy_default,buy_sell_reference,{},"wf_2020,wf_2022,wf_2023_2024",,84.918053,-79.396371,48.105317,132.077977,13.554089,False,True,False,True,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
1,risk_cap_35_15_05,medium_risk_probe,"{""buy_step_initial"": 0.15, ""buy_step_normal"": ...","wf_2020,wf_2022,wf_2023_2024",,43.271216,-37.953340,12.019654,18.308910,0.777618,False,False,False,True,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
2,low_risk_20_08_02,near_drawdown_boundary_probe,"{""buy_step_initial"": 0.08, ""buy_step_normal"": ...","wf_2020,wf_2022,wf_2023_2024",,24.732060,-21.715680,5.330153,9.658820,0.416573,False,False,False,True,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
3,wf_pass_ytd18_15_15_02,wf_drawdown_pass_best_ytd_probe,"{""buy_step_initial"": 0.15, ""buy_step_normal"": ...","wf_2020,wf_2022,wf_2023_2024",,18.514585,-16.044551,5.872611,18.308910,0.777618,True,False,False,True,False,ytd_return_below_soxlbacktest_reference;ytd_re...
4,low_risk_15_05_02,wf_drawdown_pass_low_return_floor,"{""buy_step_initial"": 0.05, ""buy_step_normal"": ...","wf_2020,wf_2022,wf_2023_2024",,19.456050,-16.864520,4.016799,6.032790,0.257069,True,False,False,True,False,ytd_return_below_soxlbacktest_reference;ytd_re...
5,floor_10_03_01,very_low_risk_floor,"{""buy_step_initial"": 0.03, ""buy_step_normal"": ...","wf_2020,wf_2022,wf_2023_2024",,11.567078,-10.353510,2.214922,6.399980,1.287081,True,False,False,True,False,ytd_return_below_soxlbacktest_reference;ytd_re...


In [7]:

for name, path in ARTIFACTS.items():
    if name.endswith("metrics") or name.endswith("summary") or name in {"candidate_daily", "candidate_trades"}:
        path.parent.mkdir(parents=True, exist_ok=True)

metrics.to_csv(ARTIFACTS["candidate_metrics"], index=False)
gate_summary.to_csv(ARTIFACTS["candidate_gate_summary"], index=False)
daily_all.to_csv(ARTIFACTS["candidate_daily"], index=False)
trades_all.to_csv(ARTIFACTS["candidate_trades"], index=False)

passing = gate_summary.loc[gate_summary["retention_gate_pass"], "candidate"].tolist()
wf_dd_pass = gate_summary.loc[gate_summary["wf_drawdown_pass"], "candidate"].tolist()
legacy_default = gate_summary.loc[gate_summary["candidate"].eq("legacy_default")].iloc[0].to_dict()
selected_risk = gate_summary.loc[gate_summary["candidate"].eq("wf_pass_ytd18_15_15_02")].iloc[0].to_dict()
low_risk = gate_summary.loc[gate_summary["candidate"].eq("low_risk_15_05_02")].iloc[0].to_dict()
summary = {
    "experiment": "soxlbacktest_qlib_walkforward_baseline_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "soxlbacktest_reference_qlib_aligned_not_promotable",
    "canonical_reference_baseline": "soxl_backtest_policy_reimplementation_v1",
    "selected_buy_sell_reference": "legacy_default",
    "selected_risk_floor": "wf_pass_ytd18_15_15_02",
    "retention_gate_pass_candidates": passing,
    "wf_drawdown_pass_candidates": wf_dd_pass,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "thresholds": thresholds,
    "legacy_default": {
        "worst_wf_max_drawdown_pct": legacy_default["worst_wf_max_drawdown_pct"],
        "ytd_return_pct": legacy_default["ytd_return_pct"],
        "failures": legacy_default["failures"],
    },
    "wf_pass_ytd18_15_15_02": {
        "worst_wf_max_drawdown_pct": selected_risk["worst_wf_max_drawdown_pct"],
        "ytd_return_pct": selected_risk["ytd_return_pct"],
        "failures": selected_risk["failures"],
    },
    "low_risk_15_05_02": {
        "worst_wf_max_drawdown_pct": low_risk["worst_wf_max_drawdown_pct"],
        "ytd_return_pct": low_risk["ytd_return_pct"],
        "failures": low_risk["failures"],
    },
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
    "next_action": "do_not_treat_soxlbacktest_risk_caps_as_final_model; use this qlib-aligned reference to score trained candidates against buy/sell timing and return gates",
}
ARTIFACTS["summary"].write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion_lines = [
    "# soxlbacktest_qlib_walkforward_baseline_v1",
    "",
    "- Decision: `soxlbacktest_reference_qlib_aligned_not_promotable`.",
    "- `legacy_default` is qlib-calendar aligned and remains the buy/sell reference baseline, but fails the walk-forward drawdown gate.",
    f"- `legacy_default` worst WF MaxDD is `{legacy_default['worst_wf_max_drawdown_pct']:.2f}%` with YTD return `{legacy_default['ytd_return_pct']:.2f}%`.",
    "- `wf_pass_ytd18_15_15_02` is the strongest low-position probe found here that passes the <20% WF drawdown gate, but its YTD return is still too low for the SOXLBacktest/current-best return gates.",
    f"- `wf_pass_ytd18_15_15_02` worst WF MaxDD is `{selected_risk['worst_wf_max_drawdown_pct']:.2f}%` with YTD return `{selected_risk['ytd_return_pct']:.2f}%`.",
    "- No screened SOXLBacktest parameter variant is promotable. Use these sidecars as reference/baseline evidence for trained candidate selection, not as final retained actors.",
    "",
    "## Sidecars",
]
conclusion_lines.extend(f"- `{rel(path)}`" for path in ARTIFACTS.values())
ARTIFACTS["conclusion"].write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
summary


{'experiment': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'created_at': '2026-06-13T10:59:05.708644+00:00',
 'decision': 'soxlbacktest_reference_qlib_aligned_not_promotable',
 'canonical_reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'selected_buy_sell_reference': 'legacy_default',
 'selected_risk_floor': 'wf_pass_ytd18_15_15_02',
 'retention_gate_pass_candidates': [],
 'wf_drawdown_pass_candidates': ['wf_pass_ytd18_15_15_02',
  'low_risk_15_05_02',
  'floor_10_03_01'],
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'thresholds': {'required_soxlbacktest_return_pct': 132.077978,
  'required_best_return_pct': 213.143462,
  'max_walkforward_drawdown_pct': 20.0,
  'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']},
 'legacy_default': {'worst_wf_max_drawdown_pct': 84.918053,
  'ytd_return_pct': 132.077977,
  'failures': 'worst_wf_max_drawdown_ge_20;ytd_return_below_current_best_alignment'},
 'wf_pass_ytd18_15_15_02': {'worst_wf_max_drawdow

## Conclusion

This card upgrades SOXLBacktest from a short-window production reference into qlib-calendar-aligned walk-forward baseline evidence. The exact legacy default is useful as a buy/sell timing reference but fails the `<20%` MaxDD walk-forward gate. A low-risk parameter floor passes the drawdown gate but fails return alignment, so the final retained model still needs a trained candidate that combines this risk control with enough upside participation.
